# 02 — 1D CNN Baseline Training

Training the 3-block 1D CNN baseline on the preprocessed MIT-BIH beat segments.  
50 epochs, Adam optimiser, class-weighted CrossEntropy loss, CosineAnnealingLR.

**Input:** `(batch, 2, 360)` — 2-lead × 1-second segments  
**Output:** 5-class logits

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'wfdb', 'PyWavelets', 'scipy', 'scikit-learn', 'tqdm'], check=True)

In [ ]:
import os, sys, json
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.dataset import get_dataloaders
from src.models.cnn_1d import CNN1D
from src.train import train_epoch, eval_epoch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load preprocessed data (run src/preprocess.py first if not available)
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')
print(f'X={X.shape}, y={y.shape}')
print(f'Class counts: {np.bincount(y)}')

train_loader, val_loader, test_loader, class_weights = get_dataloaders(
    X, y, batch_size=64
)

In [ ]:
EPOCHS = 50
model = CNN1D(n_leads=2, n_classes=5).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

os.makedirs('../results/baseline', exist_ok=True)
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc, best_epoch = 0, 0

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_acc': vl_acc},
                   '../results/baseline/cnn_best.pth')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc {tr_acc*100:.2f}%/{vl_acc*100:.2f}%')

print(f'\nBest val acc: {best_val_acc*100:.2f}% at epoch {best_epoch}')

In [ ]:
# Training curves
epochs = range(1, EPOCHS+1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, history['train_loss'], label='Train', color='#2196F3')
axes[0].plot(epochs, history['val_loss'],   label='Val',   color='#F44336')
axes[0].set_title('Loss — 1D CNN Baseline', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train', color='#2196F3')
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Val',   color='#F44336')
axes[1].set_title('Accuracy — 1D CNN Baseline', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/baseline/cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Test set accuracy
ckpt = torch.load('../results/baseline/cnn_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
_, test_acc = eval_epoch(model, test_loader, criterion, device)
print(f'\nTest Accuracy: {test_acc*100:.2f}%')